# Assignment 08: Neural Machine Translation and Attention
---

**Due Date:** Tuesday 06/29/2025 (by midnight)

**Please fill these in before submitting, just in case I accidentally mix up file names while grading**:

Name: Jane Hacker

CWID-5: (Last 5 digits of cwid)

# Introduction 

Welcome to our next assignment over Text/Sequence deep learning systems.  In this assignment you will be turning your attention to Machine Translation using an encoder / decoder, as we discussed in our class.  You are going
to implement the translator using Keras LSTM recurrent layers.  But you will be adding neural attention
mechansims by hand to your encoder / decoder, instead of using the Keras attention layers.

The model you will build here could be used to translate from one language to another, such as
translating from English to Hindi.  However, language translation requires massive datasets and
usually takes days of training on GPUs. To give you a place to experiment with these models without
using massive datasets, we will perform a simpler "date translation" task. The network will input
a date written in a variety of possible formats
(*e.g. "the 29th of August 1958", "03/30/1968", "24 JUNE 1987"*).
The network will translate them into standardized, machine readable dates
(*e.g. "1958-08-29", "1968-03-30", "1987-06-24"*).
You will create some deep networks and have them learn to output dates in the common machine-readable format YYYY-MM-DD.
 
**Instructions:**

- As with the previous assignment, you will need to create the function declarations asked for
  in `src/assg_tasks.py`.  Make sure you use
  [Python Docstrings](https://www.geeksforgeeks.org/python-docstrings/) and are generally
  following [Pep8 Python Style Guide](https://peps.python.org/pep-0008/) for your code.
- Cells with `### TESTED` comment contain unit tests that are run on your implementation.  You will
  need to uncomment the call to the unit tests, but otherwise need to stay as given in the original
  notebook.
- Likewise since you need to write your declaration of the functions asked for the tasks, don't forget
  to uncomment/add the appropriate `from assg_src include X` statements in both this notebook and
  in the `../src/test_assg_tasks.py`

**In this assignment, you will:**

- Learn more about how machine translation and the basic encoder / decoder architecture pattern
  work in practice.
- Implement by hand the neural attention mechanism described in our course materials, to add attention
  to "word" (date token) embeddings.
- Get some experience with a "text" dataset but with different normalization and tokenization requirements
  than the examples we have used in our course.


# Packages

The following imports should be all of the backages that you will need for this assignment.
We are using the Keras API in this assignment and in future assignments, so the `tensorflow` and `keras` modules
you need are now available in the notebook.

In [1]:
# assignment wide imports go here, usually all of your imports for noteboosk should
# be put up at the top here, if they were not given to you at the start of the assignment
import numpy as np
import matplotlib.pyplot as plt
from pprint import pprint
import random
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

2025-06-23 01:30:30.152790: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-06-23 01:30:30.157288: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-06-23 01:30:30.166747: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1750642230.181366    2838 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1750642230.185466    2838 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1750642230.198919    2838 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin

In [2]:
# The following ipython magic will reload changed file/modules.
# So when editing function in source code modules, you should
# be able to just rerun the cell, not restart the whole kernel.
%load_ext autoreload
%autoreload 3

The imports of the function you will write have been commented out here this time.  You will need to uncomment
the imports once you declare and write your functions here, and also in the `src/test_assg_tasks.py` file to
run the unit tests on your work.

In [3]:
# Import functions/moduls from this project.  We manually set the
# PYTHONPATH to append the location to search for this assignments
# functions to just ensure the imports are found
import sys
sys.path.append("../src")

# assignment function imports for doctests and github autograding
# these are required for assignment autograding
from assg_utils import run_unittests, run_doctests
from assg_tasks import load_dataset
from assg_tasks import plot_history
from assg_tasks import EveryNEpochs
from assg_tasks import decode_sequence
from assg_tasks import decode_sequence_attention


# Task 1: Explore Date Data and create TensorFlow Dataset

You will train your model on a dataset of 1000 human readable dates and their equivalent
standardized, machine readable dates.  We have given you the function `load_dataset()` in the
assignment tasks module to generate and load the dataset strings you need to work with.  Run the following
cell to get the data.  The dataset is actually generated randomly (though we set the random seed
so you should always see the same dataset).  You can take a look at the `load_data()` method
which uses `faker` python library to generate fake data objects, which are formatted randomly using
a list of some different date formats we want our translator to be capable of handling.

In [4]:
num_samples = 1000
text_pairs = load_dataset(num_samples)

100%|██████████| 1000/1000 [00:00<00:00, 36624.76it/s]


In [5]:
print(len)
pprint(text_pairs[:10])

<built-in function len>
[('10 feb 1993', 's1993-02-10e'),
 ('26.07.70', 's1970-07-26e'),
 ('10/12/15', 's2015-10-12e'),
 ('sunday july 27 1986', 's1986-07-27e'),
 ('saturday june 9 1990', 's1990-06-09e'),
 ('sunday september 28 1980', 's1980-09-28e'),
 ('saturday may 26 2001', 's2001-05-26e'),
 ('21 dec 1978', 's1978-12-21e'),
 ('22 nov 1976', 's1976-11-22e'),
 ('friday january 7 1994', 's1994-01-07e')]


You will be doing character level tokenization for this translation task.  The only characters that will be in
the vocabulary for the target machine readable sequences will be the digits 0-9 and the '-' character.
So we have prepended 's' and append 'e' to serve as start and end tokens for the target machine
readable date of the translation task.

Given the dataset, lets have you perform the steps you have seen in our class materials to
split the data and create TensorFlow `Dataset` object for the data, that also take care
of the tokenization and vocabulary indexing of our data.

**Task**: First in the next cell, randomly shuffle the `text_pairs` (leave the seed as given so you get same
shuffle we expect in your work).  Then split the samples into train / validation / test
samples (call these `train_pairs`, `val_pairs` and `test_pairs`).  Use 70% of the data for training, and
keep 15% for validation and testing respectively.

In [6]:
# use seed 42 to get the split we expect for your work in this assignment
# and randomly shuffle the text pairs before we select train / val / test
random.seed(42)
random.shuffle(text_pairs)

# split into train, validation and test pairs
num_val_samples = int(0.15 * len(text_pairs))
num_train_samples = len(text_pairs) - 2 * num_val_samples
num_test_samples = num_val_samples

# split the data as asked for
train_pairs = text_pairs[:num_train_samples]
val_pairs = text_pairs[num_train_samples:num_train_samples + num_val_samples]
test_pairs = text_pairs[num_train_samples + num_val_samples:]

In [7]:
print('len train_pairs:', len(train_pairs))
print('len val_pairs:', len(val_pairs))
print('len test_pairs:', len(test_pairs))

pprint(train_pairs[:5])
pprint(val_pairs[:5])

len train_pairs: 700
len val_pairs: 150
len test_pairs: 150
[('wednesday february 28 1973', 's1973-02-28e'),
 ('september 20 1970', 's1970-09-20e'),
 ('sep 18 2013', 's2013-09-18e'),
 ('22 may 2007', 's2007-05-22e'),
 ('july 10 1975', 's1975-07-10e')]
[('24 nov 2015', 's2015-11-24e'),
 ('12/22/08', 's2008-12-22e'),
 ('2 october 2024', 's2024-10-02e'),
 ('09 jan 1989', 's1989-01-09e'),
 ('27 april 1975', 's1975-04-27e')]


**Expected output**: If you have shuffled and split your data, and used the seed before shuffling, you should
get the following for your output split.

```
len train_pairs: 700
len val_pairs: 150
len test_pairs: 150
[('wednesday february 28 1973', 's1973-02-28e'),
 ('september 20 1970', 's1970-09-20e'),
 ('sep 18 2013', 's2013-09-18e'),
 ('22 may 2007', 's2007-05-22e'),
 ('july 10 1975', 's1975-07-10e')]
[('24 nov 2015', 's2015-11-24e'),
 ('12/21/08', 's2008-12-21e'),
 ('2 october 2024', 's2024-10-02e'),
 ('09 jan 1989', 's1989-01-09e'),
 ('27 april 1975', 's1975-04-27e')]
```

**Task**: Determine what the minimum, average, and maximum sequence length are in the tests data pairs
for the source and destination strings here.  This is part of our data exploration, we need to know
what sequence lengths we should use for the source and target sequences in your sequence-to-sequence models.

In [8]:
# determine the minimum, average, and maximum sequence length of 
# the source human readable strings
source_lens = [len(source) for source, target in text_pairs]
print('Min  source sequence length: ', np.min(source_lens))
print('Mean source sequence length: ', np.mean(source_lens))
print('Max  source sequence length: ', np.max(source_lens))

# determine also for the target machine readable strings
target_lens = [len(target) for source, target in text_pairs]
print('Min  target sequence length: ', np.min(target_lens))
print('Mean target sequence length: ', np.mean(target_lens))
print('Max  target sequence length: ', np.max(target_lens))

Min  source sequence length:  6
Mean source sequence length:  16.103
Max  source sequence length:  27
Min  target sequence length:  12
Mean target sequence length:  12.0
Max  target sequence length:  12


**Expected output**: Of course the target machine readable strings are all of length 12, which is what we will use as the
`target_sequence_length`.  The human readable sources range from as small as 6 to as large as 27.
For this translator we will choose a `source_sequence_length` of 30, which will no training set item
will be truncated, and all will have some padding on the end.

In [9]:
source_sequence_length = 30
target_sequence_length = 12

**Task**: Declare and implement a function named `make_date_sequence_vectorizor()`.
The function will be parameterized so that you can create the vectorization object and
adapt it to the vocabulary for both the source and destination sequences.
The function takes the following parameters:

- `sequence_length` The sequence length to pad or truncate sequences to.
- `texts` A list of the texts that will be normalized, tokenized and a vocabulary index created for.

For both of our sequences you want

- The output mode should be integer indexes, as we did for our examples in the text.
- You should only standardize by changing everything to lower case, you should not remove punctuation for these sequences.
- You want to tokenize/split by character, not by word.

The sequence length for the source (human readable dates) data will you set above.  Remember to add 1 to the
target (machine readable date) sequence length, so that when we create the tensorflow dataset we will have inputs from 0..N-2
and outputs offset by 1 timestep 1..N-1

The function should create the `TextVectorization` instance and also learn the vocabulary for the list of texts
given to your function.  The function should return the created and indexed vectorization object as its result.

In [10]:
### TESTED function make_date_sequence_vectorizer()
# uncomment when ready to run the unit tests for function
#run_unittests(['test_make_date_sequence_vectorizor'])

sequence_length = 15
texts = ['Months on ye at by esteem 04/20/1960 desire warmth former.'
         'Sure that that way gave any fond now!'
         'His boy middleton, sir nor engrossed affection excellent.'
         'Dissimilar compliment cultivated preference eat sufficient may 4th 2026.']

#vectorizor = make_date_sequence_vectorizor(sequence_length, texts)
#v = vectorizor(texts[0])
#print('vectorizor vocab size:', vectorizor.vocabulary_size())
#print(vectorizor.get_vocabulary())
#print(len(v))
#print(v)

**Expected output**: For the given text you should get the following output:

```
vectorizor vocab size: 35
['', '[UNK]', ' ', 'e', 't', 'n', 'i', 'a', 's', 'r', 'o', 'm', 'f', 'd', 'y', 'l', 'h', 'c', '0', 'w', 'u', '2', '.', 'v', 'p', 'g', 'b', '6', '4', '/', 'x', '9', '1', ',', '!']
15
tf.Tensor([11 10  5  4 16  8  2 10  5  2 14  3  2  7  4], shape=(15,), dtype=int64)
```

**Task**: When your function is working, use it to create two instances of vectorizors called
`source_vectorizor` and `target_vectorizor` in the next cell.  Remember that the sequence length of the target
vectorizor should have 1 added to it, to accommodate predicting the N+1 item.  You will need to split the pairs of sequence strings into
separate lists before you can call your function to create the vectorizor.

**Note**: It is important that the vectorizor vocabularies are only built using the training texts here.  Do you understand why?

In [11]:
# first extract the seperate source and target texts

# create vectorizors for our sequences


If your function is working and you created the source and target vectorizor instances correctly, uncommenting
and running the following cell should get the following expected output.

In [12]:
#v = source_vectorizor('wednesday february 28 1973')
#print('source vectorizor vocab size:', source_vectorizor.vocabulary_size())
#print(source_vectorizor.get_vocabulary())
#print(len(v))
#print(v)
#print()

#v = target_vectorizor('s1973-02-28e')
#print('target vectorizor vocab size:', target_vectorizor.vocabulary_size())
#print(target_vectorizor.get_vocabulary())
#print(len(v))
#print(v)

**Expected Output**: You should get the following vocabularies, sequence lengths and vectorization results
if your implementation so far is working:

```
source vectorizor vocab size: 37
['', '[UNK]', ' ', '1', '2', 'a', '0', 'e', '9', 'r', 'y', 'u', 'd', 's', '8', '7', 't', 'n', 'm', 'b', 'o', '3', '5', 'c', 'j', '4', '6', 'f', 'l', 'i', 'h', 'p', '/', 'g', '.', 'v', 'w']
30
tf.Tensor(
[36  7 12 17  7 13 12  5 10  2 27  7 19  9 11  5  9 10  2  4 14  2  3  8
 15 21  0  0  0  0], shape=(30,), dtype=int64)

target vectorizor vocab size: 15
['', '[UNK]', '-', '0', '1', '2', 's', 'e', '9', '8', '7', '3', '5', '4', '6']
13
tf.Tensor([ 6  4  8 10 11  2  3  5  2  5  9  7  0], shape=(13,), dtype=int64)
```

Now you can turn the data into `tf.data` pipeline instances as you have seen examples of.

**Task**: Declare and implement functions named `make_date_sequence_dataset()` and
`format_date_sequence_dataset()`.  These functions are about the same as the ones shown
in our text for the sequence-to-sequence learning example, so we have asked you to do
them here together.  There are a few differences:


For `make_date_sequence_dataset()` pass in the following parameters
- `pairs` A list of the sequence (source, target) text pairs, same as our text
- `batch_size` We will parameterize and set the batch size using this input parameter
- `source_vectorizor`, `target_vectorizor` Pass in the source and target vectorizor
    instances as well, and pass this along to your `format_datset()` which is where
    they are needed.

For `format_date_sequence_dataset()
- We are not going to be using `Embedding` layers in this task.  So instead, after you
  vectorize into an integer sequence, you should also then one-hot encode the sequence.
  You can (and should) use either `to_categorizal()` from Keras utils or
  TensorFlow `tf.one_hot()` to do this.
- The function should take inputs `source` and `target` which will be batches of
  tensors with strings that have not been vectorized yet.
- Also you need to pass in the `source_vectorizor` and the `target_vectorizor` instances.
- This function needs to return a `dict, array` tuple.  We are expecting the dictionary
  of inputs to have the keys `source_inputs` and `target_inputs` for this assignment.

**Hint**: To call the `format_date_sequence_dataset()` function with all of the parameters, your
call to `map()` can look something like the following:

```python
    dataset = dataset.map(lambda source, target:
                          format_date_sequence_dataset(source, target, source_vectorizor, target_vectorizor))
```

**Hint**: Don't forget to one-hot encode the sequences as mentioned.

**Hint**: Don't forget the the inputs and outputs returned for the target sequences need
correctly.  For example, since you have one-hot encoded the targets, the target inputs
should be sliced from `[:, :-1]`.  This is different from the text because now there
are 3 dimensions `(batch_no, timestep, one-hot encoding)`.  So the slice here takes all
but the last time step for the inputs.  Likewose for the target outputs, you want
all inputs from 1 to the end `[:, 1:]`.



In [13]:
### TESTED function make_date_sequence_dataset()
# uncomment when ready to run the unit tests for function
#run_unittests(['test_make_date_sequence_dataset'])

#batch_size = 25
#train_ds = make_date_sequence_dataset(train_pairs, batch_size, source_vectorizor, target_vectorizor)
#val_ds   = make_date_sequence_dataset(val_pairs, batch_size, source_vectorizor, target_vectorizor)
#test_ds  = make_date_sequence_dataset(test_pairs, batch_size, source_vectorizor, target_vectorizor)

Make sure that you uncomment the tests but also the lines to create all 3 of the datasets, we will be using them
in the following tasks.

**Expected Output**  If you care creating your datasets correctly, they should return a batch of items of size 25, and the
inputs should be a dictionary that has keys `source_inputs` and `target_inputs`.  And the shapes 
should be as shown.  e.g. You are getting batches of 25 items.  The sequence length of the source is 30 steps, and of 
the target is 12 steps.  The sources have a vocabulary size of 37 so the one-hot encoding dimension for the sources
is size 37.  The target vocabulary has 15 items, so the one-hot encoding dimension for the targets is 15.

**Note**: You can safely ignore the warning about 'calling iterator did not fully read the dataset being cached' here, that is expected when
we don't fully iterate over all items in the dataset.

```
inputs['source_inputs'].shape: (25, 30, 37)
inputs['target_inputs'].shape: (25, 12, 15)
targets.shape: (25, 12, 15)
```

In [14]:
#for inputs, targets in train_ds:
#    print(f"inputs['source_inputs'].shape: {inputs['source_inputs'].shape}")
#    print(f"inputs['target_inputs'].shape: {inputs['target_inputs'].shape}")
#    print(f"targets.shape: {targets.shape}")
#    break

# Task 2: Sequence-to-Sequence Encoder / Decoder Architecture using LSTM Layers

The example sequence-to-sequence encoder / decoder from our textbook used GRU layers and we attempted to
demonstrate it on the English to Spanish translation dataset task.

In this part of the assignment, you are going to hopefully demonstrate that the basic encoder / decoder
can work for a translation task.  The implementation of the sequence-to-sequence translator will
be similar to our text.  We are using a much simpler dataset here, translating human readable date
sequences to a different machine readable date sequence. 

We purposefully limited the dataset size to be rather small, even though we are generating the
sequences so could have made it as large as we want.  We will discuss the effects this has on what you
see later in the assignment.

**Task**: Define and implement a function named `get_lstm_translator_model()`  This will be a
function that just creates the inputs, layers and a model to define a similar Encoder RNN / Decoder
RNN architecture for our date translation task (see Figure 11.13 and Listing 11.28 and 11.29).

Your function should do the following:

- Use `LSTM` layers instead of GRU layers.  We want to use LSTM layers in the next task of the notebook for comparison
  here.
- Pass in the following parameters
  - `source_sequence_length`, `source_vocab_size`: These are the input dimensions to the encoder Input.  The source vocabulary
    size can be found from the `source_tokenizer.vocabulary_size()` method.
  - `target_sequence_length`, `target_vocab_size`: Also need the target sizes to speciffy the decoder Input shape.
  - `latent_dim` The number of units / dimensions we will use in the `LSTM` layers.

Your function should return a `keras.Model` that has the encoder source and decoder past target inputs as inputs, and the decoder
next step output prediction as the output.

There are a few other differences
- You should not use the `Embedding` dimensions in your model.  We are using the simpler one-hot
  encoding to embed the sequence token ids in a token embedding space.
- Do use a `Bidirectional` layer of `LSTM` RNNS for the encoder.  But don't use the 'sum' mode to merge the results, use the
  default concatenation instead.
- Because you use concatenation, the Bidirectional layer for the encoder will have 2 layers. If for example the `latent_dim`
  is specified to be 32, then after concatenation there will be 64 units / dimensions out from the encoder.  The decoder will
  then be using `2 * latent_dim` number of units to match the doubling of the size of the internal state.
- Also for the decoder, because `LSTM` has an internal state and an internal memory channel, you should set the initial state
  using the encoded_source for both initially, like this `initial_state=[encoded_source, encoded_source]`
- We will ask you to give the layers specific names.  Example summary and network graphs are given below, follow the same structure
  and layer names shown there (some of these are tested in the unit tests).

In [15]:
### TESTED function get_lstm_translator_model()
# uncomment when ready to run the unit tests for function
#run_unittests(['test_get_lstm_translator_model'])

In [16]:
# extract the vocabulary sizes, these represent the size of the one-hot embedding
# dimensions being used for encoder and decoder respectively
#source_vocab_size = source_vectorizor.vocabulary_size()
#target_vocab_size = target_vectorizor.vocabulary_size()

# try a model using 16 latent dimensions for both bidirectional layers in encoder, resulting in 64 dimensions in the decoder LSTM
latent_dim = 16

#model = get_lstm_translator_model(source_sequence_length, source_vocab_size, target_sequence_length, target_vocab_size, latent_dim)
#model.summary()
#keras.utils.plot_model(model, show_shapes=True, show_layer_names=True, expand_nested=True)

**Expected Output**: If you created your model as asked for, it should have the following summary and network graph shapes and names for your layers.

```
Model: "lstm_translator_model"
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ source_inputs       │ (None, 30, 37)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ target_inputs       │ (None, 12, 15)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder             │ (None, 32)        │      6,912 │ source_inputs[0]… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder (LSTM)      │ (None, 12, 32)    │      6,144 │ target_inputs[0]… │
│                     │                   │            │ encoder[0][0],    │
│                     │                   │            │ encoder[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_output      │ (None, 12, 15)    │        495 │ decoder[0][0]     │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘
 Total params: 13,551 (52.93 KB)
 Trainable params: 13,551 (52.93 KB)
 Non-trainable params: 0 (0.00 B)
```

![Expected LSTM encoder / decoder translator model](../figures/lstm_translator_model_expected.png)


**Task**: In the following cell(s), do the usual to compile, train and evaluate the encoder / decoder model on your dataset:

- For this task the `Adam` optimizer works better.  Use a learning rate of 0.005.
- The outputs are a softmax over a one-hot encoded embedding.  Do you know what loss function you should use?
- Monitor the `accuracy` as your training metric.
- Train for 200 epochs.  Validate using the validation data.
- Plot the learning curve performance on the loss and accuracy metric (`plot_history()` function is available in `assg_tasks.py`).
- Evaluate the performance of the final model on the test dataset.

In [17]:
# compile model, use the Adam optimizer with specified learning rate and correct loss function

In [18]:
# fit model for 200 epochs using validation data and

In [19]:
# plot resulting learning curves of the LSTM encoder/decoder

In [20]:
# final evaluation on test data 


**Expected Results**:

You should usually see that after 100 epochs or so of training, that you may see evidence of overfitting finally.  The
performance of validation will usually diverge.  Validation accuracy will usually stall at a bit over 92%, and that is
what you will get for the performance on the test data.

We have given an implementation of the `decode_sequence()` method (textbook listing 11.31) modified for this translation
task (in `assg_tasks.py`).  The biggest difference is that, since we are not using embedding, you have to one-hot encode the
tokenized input before feeding into the model.  But otherwise this works the same as shown in the textbook.  See how the trained
translator does so far on these or other examples you would like to try in the following cell.

In [21]:
examples = ['3 May 1979',
            '5 April 09', 
            '21th of August 2016', 
            'Tue 10 Jul 2007', 
            'Saturday May 9 2018', 
            'March 3 2001', 
            'March 3rd 2001', 
            '1 March 2001']

for example in examples:
    print('-')
    print(example)
    #decoded_example = decode_sequence(model, example, source_sequence_length, source_vectorizor, target_sequence_length, target_vectorizor)
    # chop off the start and end tokens
    #print(decoded_example[1:-1])

-
3 May 1979
-
5 April 09
-
21th of August 2016
-
Tue 10 Jul 2007
-
Saturday May 9 2018
-
March 3 2001
-
March 3rd 2001
-
1 March 2001


We have purposely used a small dataset and a fairly low number of internal states for the encoder/decoder here to get models that usually
seem to overfit and not quite have enough power to get test up to the 99% accuracy range.  It is easy with just a bit more data, or using
more latent dimensions in the RNN layers to get the basic encoder model you have here to get about the maximum performance.

But lets see, if with this same size of dataset and same number of latent dimensions in our encoder / decoder, if adding attention will help
the model perormance.

# Task 3: Custom Attention Layer

We are going to try and improve the performance of your model by adding in an attention mechanism.  You will do this by implementing
a custom Keras layer that performs the tensor operations to implement attention.  Then we will try inserting this custom layer between
the output of your encoder and decoder from your first LSTM model and see how it effects performance.

The following figure illustrates what you will implement with this custom layer.

![Neural Attention Mechanism](../figures/attn_mechanism.png)

In this figure, $s$ represents the previous output state from our decoder / attention RNN.  And $a$ represents the history of
all output from the encoder ($T_x$ is the sequence lenge of the encoder, what we are calling source_sequence_length).

Your layer starts by repeating the previous attention RNN output `source_sequence_length` times, and then concatenating those
repeated previous decoder states with the full encoder state output.  

Then the Dense and Softmax layers perform the dot product of each item with all the others to get initial weights or energies
of how similar each embedding is to all the others in the sequence context.  The softmax layer performans a normalization of these weights.

The final context is calculated as the weighted sum between the output of the weightings and the original encoder embeddings.  The context
is thus th updated embeddings that we should use instead of the original embedding at time $t$.


You will create a subclass of a Keras Layer called `AttentionLayer`.  The new layer is not too difficult, it is a sequence of several
matrix multiplciations, and the steps will be described here and tested to make sure you have the correct layers and operations.


**Task**: Create a custom Keras Layer named `AttentionLayer`.

The layer will need the following done in the `__init__` constructor:

- We need a class parameter that specifies the `sequence_length` to this attention layer, as we will need to repeat the current encoder state this number of times.
- Create a `RepeatVector` layer named `repeator`, that will repeat any vector the `sequence_length` number of times as need.
- Create a `Concatenate` layer named `concatenator` that concatenates along the last axis `axis=-1`.
- Create a `Dense` layer named `intermediate_energies`.  This should have output 10 features and should use `tanh` activation.
- Create another `Dense` layer named `energies`.  This should have a single output using `relu` activation.
- Create an `Activation` layer named `attention_weights` that uses `softmax` activation.  We need a version of the `softmax` function 
- Create a final `Dot` layer named `context`.  A `Dot` layer performs a simple dot product of some vectors.

The `__call__` method of your class will mostly operate sequentially.

- The `__call__()` method will be given `inputs` that are a dictionary.  This will have 2 keys `encoder_states` and `decoder_state`.  The `encoder_states`
  is the full state of the `encoder`, so it has a shape `(batch_size, source_sequence_length, latent_dim)`, where whatever number of dimensions was output
  from the encoder, there is 1 of those states for each of the inputs in the source sequence.
- The `decoder_state` is the single N-1 last state from the decoder.  This needs to be repeated to match the number of sequences.
- Use the `repeator` layer to repeat the single `decoder_state` into the same number of sequences as the `encoder_states`.
- Then you  need to concatenate the `encoder_states` and the repeated `decoder_states` using the `concatenator`
- The result from the concatenation can then have the neural attention calculation performed on it.  Pass to the `intermediate_energies` layer
  to perform the first dot product between each sequence item and all others (the neural weights).
- And pass to the `energies` layer which does the normalization and softmax of the neural attention we discussed
- Then pass to the `attention_weights` layer.  This performs the weighted sum of the embeddings, and is basically what needs to be applied to the
  original embeddings before attention to get slightly modified embeddings that take surrounding context.
- The final `context` layer applies the attention weights to get the final updated embedding representations.  This final context is what should be
  returned from this layer.


In [22]:
### TESTED function test_AttentionLayer()
# uncomment when ready to run the unit tests for function
#run_unittests(['test_AttentionLayer'])

#num_samples = 10
#rnn_dim = 64

# seems necessary to set seeds in base python random library, as well
# as numpy and tensorflow, if want to get same random initial weights
# in a model for testing
random.seed(10)
np.random.seed(10)
tf.random.set_seed(10)

#encoder_states = np.random.uniform(0, 1, (num_samples, source_sequence_length, rnn_dim))
#decoder_state = np.random.uniform(0, 1, (num_samples, rnn_dim))

#layer = AttentionLayer(source_sequence_length)
#context = layer({'encoder_states': encoder_states,
#                 'decoder_state': decoder_state})

#print('encoder_states.shape:', encoder_states.shape)
#print('decoder_state.shape:', decoder_state.shape)
#print('context.shape:', context.shape)
#print(context[0,0])

**Expected Output**:  Given the random seeds, you should find that, if you are performing the
neural attention calculation in your layer, that you will get the following resulting shapes and
context vector for sample 0:

```
encoder_states.shape: (10, 30, 64)
decoder_state.shape: (10, 64)
context.shape: (10, 1, 64)
tf.Tensor(
[0.5037994  0.43870404 0.55076516 0.5320272  0.4497775  0.4148697
 0.5386045  0.5177513  0.40090263 0.42972448 0.4572997  0.47683048
 0.49402452 0.536242   0.52213925 0.46703514 0.59789056 0.49478093
 0.46939936 0.542826   0.4635692  0.44132176 0.601612   0.554757
 0.49511957 0.54176253 0.58370805 0.53299934 0.49154517 0.51990664
 0.49009022 0.4908493  0.46393758 0.3727612  0.5021915  0.5757127
 0.48293594 0.6114995  0.494828   0.51572114 0.61215955 0.5642143
 0.48494598 0.43998736 0.5452579  0.5203981  0.49647036 0.4862917
 0.5122664  0.4700712  0.527557   0.4904838  0.48257697 0.5416058
 0.4744855  0.46109793 0.5608495  0.41660565 0.48337755 0.525952
 0.470785   0.48944175 0.49040702 0.5216422 ], shape=(64,), dtype=float32)
```


# Task 4: Sequence-to-Sequence Encoder with Attention

For your final task, lets try and add your new `AttentionLayer` to your model. You can start by copying your previous
`get_lstm_translator_model()` and renaming it `get_attention_translator_model()`

The basic approach is that you want to create a new `AttentionLayer` named `attention`, that will take the full output state of
the encoder, and a single sequence state from the decoder and calculate the attention for it.  You actually have to create
a loop by hand, to single step through the decoder calculating each output prediction in its sequence, using the full
encoder state and applying context.

**Task**:  Create a function named `get_attention_translator_model()`  You can start with a copy of the previous
model function, you will need all of the same inputs (plus an additional one) and will be returning a model again.

Do the following to your model.

- Add in a `batch_size` parameter as the last parameter to your function.  We have to hardcode some initial state arrays using the
  batch size we are training with.
- Create the source inputs and encoder and connect them up as before.  But modify the `Bidirectional` layer for your encoder to
  do `return_sequences=True` and return the whole sequence instead of just the final one.  We want the whole encoded sequence
  state from the encoder to perform neural attention with.
- Create an instance of your `AttentionLayer` named `attention`.  It should be set to use the `source_sequence_length` when repeating
  the state it will use.
- Don't connect up your decoder to the target inputs, nor the decoder outputs to the decoder yet, just create instances of them.
- Your decoder needs to be set to `return_state=True` so that it returns its full state.  Which as you know for an LSTM means it will
  now return a tuple when called of `(decoder_state, memory_state, carry_state)` where the decoder state is the output from the layer,
  and as you learned about before, the memory state and carry state are the two internal LSTM states of the layer.
- There is a custom layer given to you named `TfStackWrapper()`, create an instance of that layer named 'outputs'

At this point only the encoder has been connected to the source inputs.  The other layers are just instances.
We need to create a loop that iterates over the target sequence length and applies the neural attention context by hand at each time step.
The pseudocode for this is as follows:

```python
# Step 1: initialze variables to gather sequence outputs into full final sequence tensor
initialize an empty python list called outputs.
initialize 3 zero-filled numpy arrays called decoder_state, memory_state and carry_state, all of shape (batch_size, 2 * latent_dim)

# Step 2: iterate over the sequence time steps from 0 to the target_sequence_length
for t in range(target_sequence_length):

    # Step 2.A perform on calculation of attention
    Use your attention layer to calculate a new `context` using the current `encoder_states` and `decoder_state`
    The `decoder_state` is 0 for the first iteration of the loop, it contains a single state of shape `(batch_size, 2*latent_dim)`
    The `encoder_states` is the full sequence output from the encoder, it is of shape `(batch_size, source_sequence_length, 2*latent_dim)`

    # Step 2.B apply the decoder LSTM using the context from attention and the previous memory and carry states
    This step calculates the next time step using the previous context and the `memory_state` and `carry_state` from the previous time step.
    You should pass the context as the input to the decoder layer, and set the `initial_state` to pass in the `memory_state` and `carry_state`
    The decoder as we mentioned will return a tuple of the next `(decoder_state, memory_state, carry_state)` which you need to reassign to variables for next iteration

    # Step 2.C apply dense layer to decoder output
    Your layer called `decoder_output` performs the softmax on the decoder output.  Pass the `decoder_state` to it to get the prediciton for this iteration

    # Step 2.D append output
    append the output from the `decoder_output` to the python list of outputs we are gathering from iterating over the target sequence

# Step 3: stack outputs into tensor
The TfStackWrapper layer can be used to take the list of outputs and stack them into a tensor, which is what the fit method is expecting

```

Don't forget to create a Keras `Model` of the inputs and outputs named `attention_translator_model`

**TODO** **NOTE**:  This task is being adapted from an example, and it is not quite a correct implementation of Attention as we described in class.
For one, you will find that the `target_inputs` are not actually used in this model's decoder (yet it still works pretty well).  So when you
create your `Model` to return, you will only list the `source_inputs` as input to the model, not both the source and target inputs.

To fix this, the attention layer needs to be modified to be calculating attention over the target sequence representations, we are really doing
it here over the source sequence output representations (kind of).  So I leave it as an exercise for now to the motivated student to get
this to work more like the attention we described in classs.

Also we have hardcoded the batch size in the model returned by this function, by initializing the starting decoder, memory and carray
state to a particular batch size.  This is not the right way to do this, but it is complex (apparently) to get
initial arrays/vectors as needed that will defer their actual batch dimension input size correctly.  Would welcome
anyone who has an example of correct way to do this kind of initialization so we don't hardcode batch size like this in the model.

In [23]:
### TESTED function test_get_attention_translator_model()
# uncomment when ready to run the unit tests for function
#run_unittests(['test_get_attention_translator_model'])

In [24]:
# try a model using 16 latent dimensions for both bidirectional layers in encoder, resulting in 64 dimensions in the decoder LSTM
latent_dim = 16

#model = get_attention_translator_model(source_sequence_length, source_vocab_size, target_sequence_length, target_vocab_size, latent_dim, batch_size)
#model.summary()
#keras.utils.plot_model(model,  show_shapes=True, show_layer_names=True, expand_nested=True)

**Expected Output** The expected summary and network diagram for your attention translator should be as follows

```
Model: "attention_translator_model"
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ source_inputs       │ (None, 30, 37)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder             │ (None, 30, 32)    │      6,912 │ source_inputs[0]… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention           │ (25, 1, 32)       │        661 │ encoder[0][0],    │
│ (AttentionLayer)    │                   │            │ decoder[0][0],    │
│                     │                   │            │ encoder[0][0],    │
│                     │                   │            │ decoder[1][0],    │
│                     │                   │            │ encoder[0][0],    │
│                     │                   │            │ decoder[2][0],    │
│                     │                   │            │ encoder[0][0],    │
│                     │                   │            │ decoder[3][0],    │
│                     │                   │            │ encoder[0][0],    │
│                     │                   │            │ decoder[4][0],    │
│                     │                   │            │ encoder[0][0],    │
│                     │                   │            │ decoder[5][0],    │
│                     │                   │            │ encoder[0][0],    │
│                     │                   │            │ decoder[6][0],    │
│                     │                   │            │ encoder[0][0],    │
│                     │                   │            │ decoder[7][0],    │
│                     │                   │            │ encoder[0][0],    │
│                     │                   │            │ decoder[8][0],    │
│                     │                   │            │ encoder[0][0],    │
│                     │                   │            │ decoder[9][0],    │
│                     │                   │            │ encoder[0][0],    │
│                     │                   │            │ decoder[10][0],   │
│                     │                   │            │ encoder[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder (LSTM)      │ [(25, 32), (25,   │      8,320 │ attention[0][0],  │
│                     │ 32), (25, 32)]    │            │ attention[1][0],  │
│                     │                   │            │ decoder[0][1],    │
│                     │                   │            │ decoder[0][2],    │
│                     │                   │            │ attention[2][0],  │
│                     │                   │            │ decoder[1][1],    │
│                     │                   │            │ decoder[1][2],    │
│                     │                   │            │ attention[3][0],  │
│                     │                   │            │ decoder[2][1],    │
│                     │                   │            │ decoder[2][2],    │
│                     │                   │            │ attention[4][0],  │
│                     │                   │            │ decoder[3][1],    │
│                     │                   │            │ decoder[3][2],    │
│                     │                   │            │ attention[5][0],  │
│                     │                   │            │ decoder[4][1],    │
│                     │                   │            │ decoder[4][2],    │
│                     │                   │            │ attention[6][0],  │
│                     │                   │            │ decoder[5][1],    │
│                     │                   │            │ decoder[5][2],    │
│                     │                   │            │ attention[7][0],  │
│                     │                   │            │ decoder[6][1],    │
│                     │                   │            │ decoder[6][2],    │
│                     │                   │            │ attention[8][0],  │
│                     │                   │            │ decoder[7][1],    │
│                     │                   │            │ decoder[7][2],    │
│                     │                   │            │ attention[9][0],  │
│                     │                   │            │ decoder[8][1],    │
│                     │                   │            │ decoder[8][2],    │
│                     │                   │            │ attention[10][0], │
│                     │                   │            │ decoder[9][1],    │
│                     │                   │            │ decoder[9][2],    │
│                     │                   │            │ attention[11][0], │
│                     │                   │            │ decoder[10][1],   │
│                     │                   │            │ decoder[10][2]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_output      │ (25, 15)          │        495 │ decoder[0][0],    │
│ (Dense)             │                   │            │ decoder[1][0],    │
│                     │                   │            │ decoder[2][0],    │
│                     │                   │            │ decoder[3][0],    │
│                     │                   │            │ decoder[4][0],    │
│                     │                   │            │ decoder[5][0],    │
│                     │                   │            │ decoder[6][0],    │
│                     │                   │            │ decoder[7][0],    │
│                     │                   │            │ decoder[8][0],    │
│                     │                   │            │ decoder[9][0],    │
│                     │                   │            │ decoder[10][0],   │
│                     │                   │            │ decoder[11][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ outputs             │ (25, 12, 15)      │          0 │ decoder_output[0… │
│ (TfStackWrapper)    │                   │            │ decoder_output[1… │
│                     │                   │            │ decoder_output[2… │
│                     │                   │            │ decoder_output[3… │
│                     │                   │            │ decoder_output[4… │
│                     │                   │            │ decoder_output[5… │
│                     │                   │            │ decoder_output[6… │
│                     │                   │            │ decoder_output[7… │
│                     │                   │            │ decoder_output[8… │
│                     │                   │            │ decoder_output[9… │
│                     │                   │            │ decoder_output[1… │
│                     │                   │            │ decoder_output[1… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘
 Total params: 16,388 (64.02 KB)
 Trainable params: 16,388 (64.02 KB)
 Non-trainable params: 0 (0.00 B)
```

![Expected LSTM encoder / decoder Model with Attention](../figures/attention_translator_model_expected.png)

**Task**: In the following cell(s), do the usual to compile, train and evaluate the encoder / decoder model with
attention added on your dataset:

- For this task the `Adam` optimizer works better.  Use a learning rate of 0.005.
- The outputs are a softmax over a one-hot encoded embedding.  Do you know what loss function you should use?
- Monitor the `accuracy` as your training metric.
- Train for 200 epochs.  Validate using the validation data.
- Plot the learning curve performance on the loss and accuracy metric (`plot_history()` function is available in `assg_tasks.py`).
- Evaluate the performance of the final model on the test dataset.

In [25]:
# compile model here.  Use the Adam optimizer with specified learning rate and correct loss function again


In [26]:
# fig your model using same parameters for number of epochs and validation data

In [27]:
# plot the learning curves for the encoder/decoder with attention

In [28]:
# show final evaluation on test data


**Expected Results**:

The model with attention should usually converge much quicker, and validation accuracy will be pretty much
99% around epoch 100 or so.  The model does not overfit, it is learning a robust representation, and quicker
and better than the model without attention.  Training accuracy should show the model get 99% next
token prediction accuracy.


As mentioned in the TODO we also have a bit of a kludge and have hardcoded the batch size in your new
function to create your model with an attention layer.  This means we need a slightly different version of the
decode sequence function (given again for you in `assg_tasks.py`, that will reshape input tensors
as needed to have batchs of 25.  But again try out the trained model with attention on some examples, and see how
it performs.

In [29]:
examples = ['3 May 1979',
            '5 April 09', 
            '21th of August 2016', 
            'Tue 10 Jul 2007', 
            'Saturday May 9 2018', 
            'March 3 2001', 
            'March 3rd 2001', 
            '1 March 2001']

for example in examples:
    print('-')
    print(example)
    #decoded_example = decode_sequence_attention(model, example, source_sequence_length, source_vectorizor, target_sequence_length, target_vectorizor)
    # chop off the start and end tokens
    #print(decoded_example[1:-1])

-
3 May 1979
-
5 April 09
-
21th of August 2016
-
Tue 10 Jul 2007
-
Saturday May 9 2018
-
March 3 2001
-
March 3rd 2001
-
1 March 2001


# Summary / **TODO** Optional (Ungraded)

I ran out of time getting this assignment ready.  The translator with attention is not really an encoder / decoder if you implemented as asked.  In fact,
the `decoder` is not being used as the decoder in my current instructions.  It is really just a RNN layer that calculates the update
context embeddings on the source encoder output tokens.  We should rename this `neural_attention` or similar, and to make the model back into a true
encoder / decoder architecture, the output from the `neural_attention` should be fed to a new LSTM `decoder`, along with the `target_inputs` that we didn't
end up using for Task 5.  

Though as you should have seen, just using attention and performing perdictions on the output encoded sequence from the encoder is more than adequate for this tasks,
and performs a bit better usually than a decoder / decoder without attention.

# Summary

Congratulations on completing this assignment. Hopefully this has given you more experience and understanding of the important encoder / decoder pattern for doing sequence-to-sequence
learning.  These are a powerful type of Deep learning architecture, with many varied and extremely useful use cases.

<font color='blue'>
    
**What to remember from this assignment:**

- The basic encoder / decoder architecture takes inputs from a source sequence and encodes them using an RNN or transformer into an encoded state.
- The decoder of a basic encoder / decoder takes the full encoded state and the target sequence, and learns to predict the next token of the target from the given
  previous targets.
- Neural attention is a powerful mechanism to increase performance in sequence-to-sequence tasks, and other contexts.
  - It works by calculating context updates using all tokens in a sequence as the context.  The updates are a dot product of each vector with its context, giving a weighting
    score, that can be applied to update a word embedding with its contextual suroundings.